In [ ]:
#INST414 Sprint 2: Customer Churn Prediction

# improt lib
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

#load thedata
print("loading... data")
data = pd.read_csv(r'C:\Users\birha\OneDrive\Desktop\INST 414\telco_churn.csv')
print("data head")
print(data.head())

# clen the data
print("cleaning..")
data['TotalCharges'] = pd.to_numeric(data['TotalCharges'], errors='coerce')
data['TotalCharges'].fillna(data['MonthlyCharges'].median(), inplace=True)
data['Churn'] = data['Churn'].map({'Yes':1, 'No':0})
data['PaperlessBilling'] = data['PaperlessBilling'].map({'Yes':1, 'No':0})
data['TenureGroup'] = pd.cut(data['tenure'],
bins=[0, 12, 24, 36, 48, 60, 72],
labels=['0-1yr', '1-2yr', '2-3yr', '3-4yr', '4-5yr', '5+yr'])

# charts
plt.figure(figsize=(8,5))
sns.countplot(x='Churn', data=data)
plt.title("churn distrib")
plt.show()

plt.figure(figsize=(8,5))
sns.boxplot(x='Churn', y='MonthlyCharges', data=data)
plt.title("charges by churn")
plt.show()

plt.figure(figsize=(10,5))
sns.countplot(x='TenureGroup', hue='Churn', data=data)
plt.title("tenure group")
plt.xticks(rotation=45)
plt.show()

plt.figure(figsize=(8,6))
sns.heatmap(data.corr(numeric_only=True), annot=True)
plt.title("correaltions")
plt.show()

#Hypothesis
print("High pay churn:", data[data['MonthlyCharges'] > 70]['Churn'].mean())
print("Low pay churn:", data[data['MonthlyCharges'] <= 70]['Churn'].mean())

# make  model
print("making model")
features = ['MonthlyCharges', 'tenure', 'TotalCharges', 'PaperlessBilling']
X = pd.get_dummies(data[features])
y = data['Churn']

split_point = int(0.8 * len(X))
X_train = X[:split_point]
X_test = X[split_point:]
y_train = y[:split_point]
y_test = y[split_point:]


#logestic
logreg = LogisticRegression(max_iter=1000)
logreg.fit(X_train, y_train)
log_pred = logreg.predict(X_test)
print("Logistic acc:", accuracy_score(y_test, log_pred))

#rf
rf = RandomForestClassifier()
rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)
print("RF acc:", accuracy_score(y_test, rf_pred))

# result
print("churn rate:", data['Churn'].mean())
print("best acc:", accuracy_score(y_test, rf_pred))

#save
import os
os.makedirs('data/processed', exist_ok=True)
print("saving file")
data.to_csv('data/processed/churn_analysis_results.csv', index=False)
print("done")
